In [4]:
import os
import polars as pl
from pathlib import Path

In [50]:
E_FACTOR = [0.8]
H_FACTOR = ["0.1"]
C_FACTOR = [
    0.25,
    0.5,
    1,
    3,
]
POLYMORPHISM = ["verylow", "low", "high"]

TREES = 16
REPLICAS = 4

OUTPUT_FOLDER = "simulated_data"

N_HP_STATES = 1
HP_STATES = [str(i) for i in range(N_HP_STATES)]

PCH = Path(os.path.abspath("")).parent.parent

In [51]:
def get_single_row_stats(
    id: str, feature: str, weight: str, chrmap: dict[str, list[str]]
):
    is_morph = id[0] == "m"
    has_hp = any("0" in v for v in chrmap.values())
    return {"is_morph": is_morph, "has_hp": has_hp}

In [62]:
def get_single_dataset_stats(
    csv_path: Path,
):
    dataset = pl.read_csv(csv_path, infer_schema=False)

    row_stats = []

    for row in dataset.iter_rows(named=True):
        chrmap = {
            k: str(v).split("/")
            for k, v in row.items()
            if k not in ["id", "feature", "weight"]
        }
        row_stats.append(
            get_single_row_stats(
                id=row["id"],
                feature=row["feature"],
                weight=row["weight"],
                chrmap=chrmap,
            )
        )
    return pl.DataFrame(row_stats)

In [63]:
get_single_dataset_stats(PCH / "data/simulated_data/verylow_0.1_0.8_1/sim_tree1_2.csv")

is_morph,has_hp
bool,bool
true,true
true,false
true,true
true,true
true,false
…,…
false,false
false,false
false,false


In [64]:
from tqdm import tqdm
from itertools import product

In [95]:
different_conditions_data = []
for ef, hf, cf, poly in tqdm(product(E_FACTOR, H_FACTOR, C_FACTOR, POLYMORPHISM)):
    for tree, rep in product(range(1, 17), range(1, 5)):
        dataset_stats = get_single_dataset_stats(
            PCH / f"data/simulated_data/{poly}_{hf}_{ef}_{cf}/sim_tree{tree}_{rep}.csv"
        ).with_columns(
            poly=pl.lit(poly),
            ef=pl.lit(ef),
            hf=pl.lit(hf),
            cf=pl.lit(cf, dtype=pl.Float32),
            tree=pl.lit(tree),
            rep=pl.lit(rep),
        )
        different_conditions_data.append(dataset_stats)

12it [00:30,  2.57s/it]


In [96]:
result_df = pl.concat(different_conditions_data)

In [97]:
result_df

is_morph,has_hp,poly,ef,hf,cf,tree,rep
bool,bool,str,f64,str,f32,i32,i32
true,false,"""verylow""",0.8,"""0.1""",0.25,1,1
true,false,"""verylow""",0.8,"""0.1""",0.25,1,1
true,false,"""verylow""",0.8,"""0.1""",0.25,1,1
true,false,"""verylow""",0.8,"""0.1""",0.25,1,1
true,false,"""verylow""",0.8,"""0.1""",0.25,1,1
…,…,…,…,…,…,…,…
false,false,"""high""",0.8,"""0.1""",3.0,16,4
false,false,"""high""",0.8,"""0.1""",3.0,16,4
false,false,"""high""",0.8,"""0.1""",3.0,16,4


In [99]:
# for each poly level, what is the percentage of is morph?

In [100]:
agg_df = result_df.group_by("poly").agg(
    n_morph=pl.col("is_morph").cast(pl.Int64).sum(),
    n_no_morph=((~pl.col("is_morph")).cast(pl.Int64).sum()),
    n_hp_and_morph=(pl.col("has_hp") & pl.col("is_morph")).sum(),
    n_hp_no_morph=(pl.col("has_hp") & ~pl.col("is_morph")).sum(),
    # pct_hp_morph=(pl.col("n_hp_and_morph") / pl.col("n_morph")),
    # pct_hp_no_morph=(pl.col("n_hp_and_no_morph") / pl.col("n_no_morph")),
)

In [106]:
agg_df = agg_df.with_columns(
    pct_hp_and_morph=pl.col("n_hp_and_morph") / pl.col("n_morph"),
    pct_hp_no_morph=pl.col("n_hp_no_morph") / pl.col("n_no_morph"),
)

In [109]:
print(agg_df.select("poly", "pct_hp_and_morph", "pct_hp_no_morph"))

shape: (3, 3)
┌─────────┬──────────────────┬─────────────────┐
│ poly    ┆ pct_hp_and_morph ┆ pct_hp_no_morph │
│ ---     ┆ ---              ┆ ---             │
│ str     ┆ f64              ┆ f64             │
╞═════════╪══════════════════╪═════════════════╡
│ high    ┆ 0.108059         ┆ 0.087039        │
│ verylow ┆ 0.108224         ┆ 0.108761        │
│ low     ┆ 0.103618         ┆ 0.110757        │
└─────────┴──────────────────┴─────────────────┘


In [108]:
print(agg_df)

shape: (3, 7)
┌─────────┬─────────┬────────────┬────────────────┬───────────────┬────────────────┬───────────────┐
│ poly    ┆ n_morph ┆ n_no_morph ┆ n_hp_and_morph ┆ n_hp_no_morph ┆ pct_hp_and_mor ┆ pct_hp_no_mor │
│ ---     ┆ ---     ┆ ---        ┆ ---            ┆ ---           ┆ ph             ┆ ph            │
│ str     ┆ i64     ┆ i64        ┆ u32            ┆ u32           ┆ ---            ┆ ---           │
│         ┆         ┆            ┆                ┆               ┆ f64            ┆ f64           │
╞═════════╪═════════╪════════════╪════════════════╪═══════════════╪════════════════╪═══════════════╡
│ high    ┆ 6080    ┆ 91200      ┆ 657            ┆ 7938          ┆ 0.108059       ┆ 0.087039      │
│ verylow ┆ 6080    ┆ 91200      ┆ 658            ┆ 9919          ┆ 0.108224       ┆ 0.108761      │
│ low     ┆ 6080    ┆ 91200      ┆ 630            ┆ 10101         ┆ 0.103618       ┆ 0.110757      │
└─────────┴─────────┴────────────┴────────────────┴───────────────┴──────────

In [102]:
760 * 15

11400